# Nephron position or tissue location: what sets injury between tubules (notebook 96)

**The question.** Our AKI analyses describe injury along one coordinate, the tubule's position along the PT. In
ischemia–reperfusion, injury is most apparent in S3, outer-medullary blood flow falls disproportionately and its
capillaries are prone to occlusion (Bonventre & Yang 2011). Beyond nephron position, does a tubule's place in the
tissue — its depth and its neighbours — explain its injury?

**The frozen protocol** is `docs/results/pt-injury-tissue-location.md` (commit 2f8167f).
- A1 (H1, recovery): injury rises with nephron position.
- A2 (H2): at fixed position, deeper tubules (farther from glomeruli) are more injured, beyond controls.
- A3 (H3): injury residuals cluster among neighbouring tubules beyond position and depth.
- D4 (descriptive): graded or all-or-none within position.
- Sensitivity: the injured S3 units the package dropped (notebook 76's placed c15 and c19 units) added back.

**Sections.** 1 setup and inputs; 2 scores, depth and effects; 3 A1–A3 and D4; 4 sensitivity; 5 verdicts;
6 figures; 7 reading and run record.

## 1 · Setup and inputs

In [ ]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
EXPORT = results_root / 'package_pseudospace' / 'aki' / 'pt_units.h5ad'
RUN = results_root / 'package_pseudospace' / 'runs' / 'aki'
T76 = results_root / 'pt_package_aki' / 'tables'
output = results_root / 'pt_injury_tissue_location'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '96.tissue_location.1'
SEED = 96
FINGERPRINT = 'bb4a80c1b484'
CONTROLS, IRS = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
SPECIMENS = CONTROLS + IRS
SD_FLOOR, BIN_WIDTH, N_STRATA, SPLINE_DF, K_NEIGHBOURS, K_GLOM = .25, .05, 20, 5, 6, 3
N_PERM = 1000
H2_MARGIN, H3_MARGIN, BIC_MARGIN = .25, .05, 10.
CORRELOGRAM = (3, 6, 12, 24)
PROGRAM_SIGN = {'INJ': 1, 'TPT': -1, 'CYC': 1}
REQUIRED = [EXPORT, RUN / '03_integration' / 'pass1_labelled.h5ad', T76 / 'positive_control_markers.csv',
            T76 / 'programs_transporters.csv', T76 / 'gene_sets.json', T76 / 'universe.csv', T76 / 'placement_units.csv',
            *[RUN / '01_units' / f'{s}_units.h5ad' for s in SPECIMENS]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70 and 76 first)')

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy import sparse
from scipy.stats import spearmanr
from IPython.display import display

from pseudospace.package_inputs import read_pt_units, support_positions
from pseudospace.pathway_networks import segment_cuts
from pseudospace.tissue_location import (depth_effect, equal_count_bins, glomerular_depth, knn_weights, mixture_bic,
                                         moran_null, morans_i, position_expectation, program_score)

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'IR2A2': '#D55E00', 'IR2A4': '#E69F00'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')

### 1.1 · Units, programs and guards (not cached)

- G1: the export's pass-1 fingerprint is notebook 70's; the common support is notebook 76's (14,048 units).
- G2: the programs have notebook 76's sizes (18 markers, T_PT 93, C 297).

In [ ]:
guard_log = []
units = read_pt_units(EXPORT, 'aki')
assert units.uns['provenance']['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT, 'G1: fingerprint'
obs = units.obs
mask, position = support_positions(obs.pseudospace.to_numpy(float), obs['sample'].astype(str).to_numpy(),
                                   obs.segment_class, rescale='support')
raw = obs.pseudospace.to_numpy(float)
LO, HI = float(raw[mask].min()), float(raw[mask].max())
assert int(mask.sum()) == 14048, f'G1: {int(mask.sum())} units in the common support'
guard_log.append(f'G1: export fingerprint {FINGERPRINT}; 14,048 units in the common support')
GENES = pd.Index(units.var_names.astype(str))
markers = pd.read_csv(T76 / 'positive_control_markers.csv', index_col=0).index.astype(str).tolist()
sets76 = json.loads((T76 / 'gene_sets.json').read_text())
u76 = set(pd.read_csv(T76 / 'universe.csv').gene.astype(str))
cyc = sorted((set(sets76['MSigDB_Hallmark_2020::G2-M Checkpoint']) | set(sets76['MSigDB_Hallmark_2020::E2F Targets'])) & u76)
tr = pd.read_csv(T76 / 'programs_transporters.csv')
tpt = sorted(tr.mouse[tr.in_T_PT.astype(bool)].astype(str))
PROGRAMS = {'INJ': markers, 'TPT': tpt, 'CYC': cyc}
assert (len(markers), len(tpt), len(cyc)) == (18, 93, 297), 'G2: program sizes'
assert all(set(v) <= set(GENES) for v in PROGRAMS.values()), 'G2: program genes missing from the export'
guard_log.append("G2: programs INJ 18, TPT 93, CYC 297 (notebook 76's), all in the export")
print('\n'.join('· ' + g for g in guard_log))

U = pd.DataFrame({'specimen': obs['sample'].astype(str).to_numpy()[mask], 'x': position,
                  'segment': obs.segment_class.astype(str).to_numpy()[mask],
                  'cx': obs.x_centroid.to_numpy(float)[mask], 'cy': obs.y_centroid.to_numpy(float)[mask],
                  'arm': 'export'}, index=obs.index[mask].astype(str))
COUNTS = sparse.csr_matrix(units.X[mask])
del units

# sensitivity units: notebook 76's placed c15 and c19 units on the PT stretch
placed = pd.read_csv(T76 / 'placement_units.csv')
placed.columns = ['cluster', 'unit', *placed.columns[2:]]
placed = placed[placed.on_pt_stretch.astype(bool)].copy()
placed['x'] = (placed.median_value.astype(float) - LO) / (HI - LO)
placed = placed[placed.x.between(0, 1)].drop_duplicates('unit').set_index('unit')
with h5py.File(RUN / '03_integration' / 'pass1_labelled.h5ad', 'r') as f:
    p1_index = pd.Index(f['obs']['_index'][:].astype(str))
    p1_genes = pd.Index(f['var']['_index'][:].astype(str))
p1 = ad.read_h5ad(RUN / '03_integration' / 'pass1_labelled.h5ad', backed='r')
rows = p1_index.get_indexer(placed.index)
assert (rows >= 0).all() and list(p1_genes) == list(GENES)
order = np.argsort(rows)
extra_counts = sparse.csr_matrix(p1.layers['counts'][np.sort(rows)])
inv = np.empty_like(order)
inv[order] = np.arange(len(order))
extra_counts = extra_counts[inv]
p1o = p1.obs.iloc[rows]
C12, C23 = segment_cuts(U.x[U.specimen.isin(CONTROLS)].to_numpy(), U.segment[U.specimen.isin(CONTROLS)].str.replace('PT-', '').to_numpy())
EXTRA = pd.DataFrame({'specimen': p1o['sample'].astype(str).to_numpy(), 'x': placed.x.to_numpy(),
                      'segment': np.where(placed.x.to_numpy() >= C23, 'placed (S3 side)', 'placed'),
                      'cx': p1o.x_centroid.to_numpy(float), 'cy': p1o.y_centroid.to_numpy(float), 'arm': 'placed ' + placed.cluster.to_numpy()},
                     index=placed.index)
del p1
print(U.groupby('specimen').size().to_dict(), '· placed units on the stretch:', EXTRA.groupby(['specimen', 'arm']).size().to_dict())

## 2 · Scores, depth and injury effects

In [ ]:
def expression(counts, genes):
    lib = np.asarray(counts.sum(axis=1)).ravel()
    block = counts[:, GENES.get_indexer(genes)].toarray()
    return np.log1p(block / lib[:, None] * 1e4)


def build(arms):
    """Scores (reference = export control units), depth, strata and effects for the units of ``arms``."""
    frame = pd.concat([U, EXTRA]) if 'placed' in arms else U.copy()
    counts = sparse.vstack([COUNTS, extra_counts]).tocsr() if 'placed' in arms else COUNTS
    ref = (frame.arm.eq('export') & frame.specimen.isin(CONTROLS)).to_numpy()
    builders = frame.arm.eq('export').to_numpy()
    for name, genes in PROGRAMS.items():
        frame[name] = program_score(expression(counts, genes), ref, floor=SD_FLOOR)
        frame[f'e {name}'] = frame[name] - position_expectation(frame.x, frame[name], frame.specimen, CONTROLS,
                                                                width=BIN_WIDTH, cross_fit=True, builders=builders)
    frame['depth'] = np.nan
    frame['depth pct'] = np.nan
    frame['stratum'] = -1
    for spec in SPECIMENS:
        o1 = ad.read_h5ad(RUN / '01_units' / f'{spec}_units.h5ad', backed='r').obs
        glom = o1[o1.seg_class.astype(str).eq('glomerulus')][['x_centroid', 'y_centroid']].to_numpy(float)
        m = frame.specimen.eq(spec).to_numpy()
        d, pct = glomerular_depth(frame.loc[m, ['cx', 'cy']].to_numpy(), glom, k=K_GLOM)
        frame.loc[m, 'depth'] = d
        frame.loc[m, 'depth pct'] = pct
        frame.loc[m, 'stratum'] = equal_count_bins(frame.x[m].to_numpy(), N_STRATA)
    return frame


t0 = time.perf_counter()
PRIMARY = build(['export'])
SENS = build(['export', 'placed'])
PRIMARY.to_csv(tables / 'units.csv')
print(f'Scores and depth {time.perf_counter() - t0:.0f} s')
display(PRIMARY.groupby('specimen')[['INJ', 'TPT', 'CYC', 'e INJ', 'e TPT', 'e CYC']].mean().round(3))
SPREAD = pd.DataFrame({s: {'units': int(PRIMARY.specimen.eq(s).sum()),
                           'median SD of depth pct within position bins': float(PRIMARY[PRIMARY.specimen.eq(s)].groupby('stratum')['depth pct'].std().median()),
                           'Spearman x vs depth': spearmanr(PRIMARY.x[PRIMARY.specimen.eq(s)], PRIMARY.depth[PRIMARY.specimen.eq(s)])[0]}
                       for s in SPECIMENS}).T
SPREAD.to_csv(tables / 'depth_spread.csv')
display(SPREAD.round(3))

## 3 · A1–A3 and D4

In [ ]:
def analyse(frame, label):
    a1, a2, a3, cg, d4, resid = [], [], [], [], [], {}
    for name in PROGRAMS:
        for spec in SPECIMENS:
            f = frame[frame.specimen.eq(spec)]
            e, x, dpt, st = f[f'e {name}'].to_numpy(), f.x.to_numpy(), f['depth pct'].to_numpy(), f.stratum.to_numpy()
            a1.append({'arm': label, 'program': name, 'specimen': spec, 'units': len(f), 'ρ(e, x)': spearmanr(e, x)[0]})
            de = depth_effect(e, x, dpt, st, n_perm=N_PERM, seed=SEED, df=SPLINE_DF)
            row = {'arm': label, 'program': name, 'specimen': spec, 'scope': 'all', 'units': len(f), 'beta': de['beta'],
                   'partial R2': de['partial R2'], 'p': de['p'], 'null low': float(np.quantile(de['null'], .025)),
                   'null high': float(np.quantile(de['null'], .975))}
            a2.append(row)
            s3 = f.segment.eq('PT-S3').to_numpy() | f.segment.eq('placed (S3 side)').to_numpy()
            if s3.sum() >= 100:
                de3 = depth_effect(e[s3], x[s3], dpt[s3], equal_count_bins(x[s3], N_STRATA), n_perm=N_PERM, seed=SEED, df=SPLINE_DF)
                a2.append({**row, 'scope': 'S3', 'units': int(s3.sum()), 'beta': de3['beta'], 'partial R2': de3['partial R2'],
                           'p': de3['p'], 'null low': float(np.quantile(de3['null'], .025)),
                           'null high': float(np.quantile(de3['null'], .975))})
            r = de['residual']
            resid[(name, spec)] = pd.Series(r, index=f.index)
            xy = f[['cx', 'cy']].to_numpy()
            for k in CORRELOGRAM:
                W = knn_weights(xy, k)
                I = morans_i(r, W)
                null = moran_null(r, W, st, n_perm=N_PERM, seed=SEED)
                entry = {'arm': label, 'program': name, 'specimen': spec, 'k': k, 'I': I, 'null mean': float(null.mean()),
                         'null high': float(np.quantile(null, .975)), 'p': float((1 + (null >= I).sum()) / (1 + N_PERM))}
                cg.append(entry)
                if k == K_NEIGHBOURS:
                    a3.append(entry)
            if name == 'INJ':
                scopes = {'S3': s3, **{f'x tertile {i + 1}': (x >= np.quantile(x, i / 3)) & (x <= np.quantile(x, (i + 1) / 3))
                                       for i in range(3)}}
                for scope, m in scopes.items():
                    b = mixture_bic(e[m], seed=SEED)
                    d4.append({'arm': label, 'specimen': spec, 'scope': scope, 'units': int(m.sum()), **b,
                               'two components preferred': b['bic 2'] < b['bic 1'] - BIC_MARGIN})
    return tuple(pd.DataFrame(t) for t in (a1, a2, a3, cg, d4)) + (resid,)


t0 = time.perf_counter()
A1, A2, A3, CG, D4, RESID = analyse(PRIMARY, 'primary')
A1s, A2s, A3s, CGs, D4s, _ = analyse(SENS, 'with placed units')
print(f'Analyses {time.perf_counter() - t0:.0f} s')
for t, n in ((A1, 'a1_position'), (A2, 'a2_depth'), (A3, 'a3_neighbours'), (CG, 'a3_correlogram'), (D4, 'd4_mixtures')):
    t.to_csv(tables / f'{n}.csv', index=False)
pd.concat([A1s.assign(table='A1'), A2s.assign(table='A2'), A3s.assign(table='A3'), D4s.assign(table='D4')]).to_csv(
    tables / 'sensitivity.csv', index=False)
display(A1.pivot(index='specimen', columns='program', values='ρ(e, x)').round(3))
display(A2[A2.scope.eq('all')].pivot(index='specimen', columns='program', values=['beta', 'p']).round(3))
display(A2[A2.scope.eq('S3')].pivot(index='specimen', columns='program', values=['beta', 'p']).round(3))
display(A3.pivot(index='specimen', columns='program', values=['I', 'null high', 'p']).round(3))
display(D4.round(2))

## 4 · Verdicts (protocol section 3), primary and with the placed units

In [ ]:
def verdicts(a1, a2, a3, label):
    out = []
    inj1 = a1[a1.program.eq('INJ')].set_index('specimen')['ρ(e, x)']
    out.append({'arm': label, 'hypothesis': 'H1 nephron position', 'holds': bool((inj1[IRS] > 0).all()),
                'values': ', '.join(f'{s} {inj1[s]:.3f}' for s in SPECIMENS)})
    b = a2[a2.program.eq('INJ') & a2.scope.eq('all')].set_index('specimen')
    h2 = bool(((b.beta[IRS] > 0) & (b.p[IRS] <= .05)).all() and all(b.beta[i] - b.beta[c] >= H2_MARGIN for i in IRS for c in CONTROLS))
    out.append({'arm': label, 'hypothesis': 'H2 depth beyond position', 'holds': h2,
                'values': ', '.join(f'{s} β {b.beta[s]:.3f} (p {b.p[s]:.3f})' for s in SPECIMENS)})
    m = a3[a3.program.eq('INJ')].set_index('specimen')
    h3 = bool((m.I[IRS] > m['null high'][IRS]).all() and all(m.I[i] - m.I[c] >= H3_MARGIN for i in IRS for c in CONTROLS))
    out.append({'arm': label, 'hypothesis': 'H3 neighbours beyond position and depth', 'holds': h3,
                'values': ', '.join(f'{s} I {m.I[s]:.3f} (null 97.5% {m["null high"][s]:.3f})' for s in SPECIMENS)})
    return out


VERD = pd.DataFrame(verdicts(A1, A2, A3, 'primary') + verdicts(A1s, A2s, A3s, 'with placed units'))
robust = {h: VERD[VERD.hypothesis.eq(h)].holds.nunique() == 1 for h in VERD.hypothesis.unique()}
VERD['sensitivity'] = VERD.hypothesis.map(lambda h: 'robust' if robust[h] else 'changes')
VERD.to_csv(tables / 'verdicts.csv', index=False)
display(VERD)

## 5 · Figures

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(180 * MM, 45 * MM), sharey=True)
for ax, spec in zip(axes, SPECIMENS):
    f = PRIMARY[PRIMARY.specimen.eq(spec)]
    sc = ax.scatter(f.x, f['e INJ'], c=f['depth pct'], s=1.2, cmap='viridis', lw=0, rasterized=True)
    bins = pd.cut(f.x, np.linspace(0, 1, 21))
    mean = f.groupby(bins, observed=True)['e INJ'].mean()
    ax.plot([b.mid for b in mean.index], mean.to_numpy(), color=INK, lw=1)
    ax.axhline(0, color=MUTED, lw=.5)
    ax.set_title(spec)
    ax.set_xlabel('PT position')
axes[0].set_ylabel('injury effect (INJ)')
fig.colorbar(sc, ax=axes, shrink=.8, label='depth percentile')
save(fig, '96a_injury_along_pt')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(120 * MM, 50 * MM))
for ax, (table, col, lab) in zip(axes, ((A2[A2.scope.eq('all')], 'beta', 'β depth (score units)'), (A3, 'I', "Moran's I of residual"))):
    for j, prog in enumerate(PROGRAMS):
        b = table[table.program.eq(prog)].set_index('specimen').reindex(SPECIMENS)
        for i, spec in enumerate(SPECIMENS):
            y = i + (j - 1) * .22
            low = b.loc[spec, 'null low'] if 'null low' in b else b.loc[spec, 'null mean']
            ax.plot([low, b.loc[spec, 'null high']], [y, y], color=LIGHT, lw=2.5, solid_capstyle='butt')
            ax.plot(b.loc[spec, col], y, ['o', 's', '^'][j], color=SPEC_COLOR[spec], ms=3.5, mfc=SPEC_COLOR[spec] if prog == 'INJ' else 'white')
    ax.axvline(0, color=MUTED, lw=.5)
    ax.set_yticks(range(len(SPECIMENS)), SPECIMENS)
    ax.invert_yaxis()
    ax.set_xlabel(lab)
handles = [plt.Line2D([], [], marker=m, ls='', color=MUTED, mfc=MUTED if p == 'INJ' else 'white', label=p)
           for m, p in zip(['o', 's', '^'], PROGRAMS)]
axes[1].legend(handles=handles, loc='lower right')
fig.tight_layout()
save(fig, '96b_depth_and_neighbours')
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(180 * MM, 50 * MM))
lim = np.nanpercentile(np.abs(np.concatenate([RESID[('INJ', s)].to_numpy() for s in SPECIMENS])), 97)
for ax, spec in zip(axes, SPECIMENS):
    f = PRIMARY[PRIMARY.specimen.eq(spec)]
    r = RESID[('INJ', spec)].reindex(f.index)
    ax.scatter(f.cx, -f.cy, c=r, cmap='RdBu_r', vmin=-lim, vmax=lim, s=1.5, lw=0, rasterized=True)
    ax.set_aspect('equal')
    ax.set_title(spec)
    ax.axis('off')
save(fig, '96c_residual_maps')
plt.show()

fig, ax = plt.subplots(figsize=(70 * MM, 50 * MM))
for spec in SPECIMENS:
    b = CG[CG.program.eq('INJ') & CG.specimen.eq(spec)]
    ax.plot(b.k, b.I, marker='o', ms=3, color=SPEC_COLOR[spec], ls='-' if spec in IRS else '--', label=spec)
    ax.plot(b.k, b['null high'], color=SPEC_COLOR[spec], lw=.5, ls=':')
ax.set_xscale('log', base=2)
ax.set_xticks(CORRELOGRAM, [str(k) for k in CORRELOGRAM])
ax.set_xlabel('neighbours (k)')
ax.set_ylabel("Moran's I, INJ residual")
ax.legend()
save(fig, '96d_correlogram')
plt.show()

## 6 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'Unit library = counts over all 19,059 export genes (the pass-1 object has the same genes) for both arms.'),
    ('reading', 'Control expectations are built from export control units only; added control units are cross-fitted like the export ones.'),
    ('reading', "The S3 scope of the sensitivity arm counts placed units beyond the controls' S2/S3 cut (segment_cuts) as S3 (they carry no package label)."),
    ('reading', 'Correlogram nulls permute within the same position bins as A3.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUNREC = {'notebook': '96_pt_injury_tissue_location', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED,
          'guards': guard_log, 'units': PRIMARY.groupby('specimen').size().to_dict(),
          'placed units': EXTRA.groupby('specimen').size().to_dict(),
          'verdicts': VERD[['arm', 'hypothesis', 'holds']].to_dict('records'),
          'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUNREC, indent=2, default=str))
print(json.dumps(RUNREC, indent=2, default=str))